# Fine-tune MobileCount + Mask trên nhãn Dublin Temple Bar

Mục tiêu: tinh chỉnh mô hình đếm người đã train trên ShanghaiTech B bằng 300 ảnh camera Temple Bar đã gán nhãn thủ công, để đếm đúng hơn trên chính camera dùng trong demo.

**Cách làm**
- **Chia theo ngày** (ảnh cùng ngày rất giống nhau, chia ngẫu nhiên sẽ làm kết quả đẹp giả): train 12–16/11, val 17/11, test 18–19/11. Test chỉ dùng một lần ở cuối.
- **Ảnh 960×540**, đúng kích thước demo đưa vào mô hình. Nhãn điểm đã nhân tỉ lệ sẵn trong dataset.
- **Vùng bỏ qua** (`ignore`): loss bằng 0 trong vùng này; điểm nằm trong vùng bị loại khỏi số đếm thật; khi đo MAE cũng không cộng density dự đoán trong vùng đó.
- **Khởi tạo từ checkpoint ShanghaiTech B**, learning rate nhỏ, đóng băng encoder vài epoch đầu.
- **Trộn ShanghaiTech B** vào mỗi batch để mô hình không quên (đo lại MAE trên ShanghaiTech B test sau khi fine-tune).
- Thử hai giá trị sigma của Gaussian (ảnh Dublin có đầu người nhỏ hơn ShanghaiTech), chọn theo val MAE.

**Add Input trên Kaggle:** (1) dataset nhãn Dublin (thư mục `kaggle/dublin_dataset`), (2) ShanghaiTech, (3) model chứa checkpoint `mc_mask_f4_partB.pth` (hoặc file `.pth` của mask model).

In [ ]:
import contextlib, copy, glob, json, math, os, random, re, time

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scipy.io as sio
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
from IPython.display import display
from matplotlib.path import Path as MplPath
from PIL import Image, ImageDraw
from torch.utils.data import DataLoader, Dataset

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('PyTorch', torch.__version__, '| torchvision', torchvision.__version__)
if device.type == 'cuda':
    prop = torch.cuda.get_device_properties(0)
    GPU_NAME = prop.name
    print(f'GPU: {prop.name}, {prop.total_memory / 1024**3:.1f} GB')
else:
    GPU_NAME = 'CPU'
    print('CẢNH BÁO: không có GPU. Vào Settings → Accelerator → chọn GPU.')
plt.rcParams.update({'figure.dpi': 100, 'axes.grid': True, 'grid.alpha': 0.3})

In [ ]:
# ---- Đường dẫn ----
INPUT_DIR, WORK_DIR = '/kaggle/input', '/kaggle/working'
CKPT_NAME_HINT = 'mc_mask_f4_partB'     # tên file checkpoint gốc; không thấy thì lấy file .pth đầu tiên có nhánh mask
# ---- Chia theo ngày (UTC; Dublin tháng 11 = UTC) ----
TEST_DAYS, VAL_DAYS = ['2026-11-18', '2026-11-19'], ['2026-11-17']   # còn lại là train
NIGHT_HOURS = (7, 17)       # giờ nằm ngoài [7, 17) tính là ban đêm
# ---- Nhãn ----
SIGMAS = [8, 15]            # thử từng sigma Gaussian (pixel, ảnh 960×540), chọn theo val MAE
SHB_SIGMA = 15              # sigma gốc khi train ShanghaiTech B
DENSITY_SCALE, STRIDE, MASK_THRESH = 100.0, 4, 1e-6   # giống notebook train gốc
# ---- Fine-tune ----
EPOCHS = 40
REPEAT = 4                  # mỗi epoch lấy mỗi ảnh Dublin REPEAT lần (mỗi lần một patch ngẫu nhiên khác)
DUB_BS, SHB_BS = 4, 4       # mỗi batch: 4 patch Dublin + 4 patch ShanghaiTech B (SHB_BS = 0 để tắt trộn)
CROP = 512
LR, LR_MIN, WEIGHT_DECAY = 2e-5, 1e-6, 1e-4
FREEZE_EPOCHS = 5           # đóng băng encoder MobileNetV2 ở các epoch đầu
MASK_LAMBDA = 0.1
VAL_EVERY = 2
NUM_WORKERS = 4
SEED = 42
QUICK = False               # True: chạy thử vài phút để kiểm tra lỗi

if QUICK:
    EPOCHS, REPEAT, SIGMAS, VAL_EVERY, FREEZE_EPOCHS = 2, 1, [8], 1, 1

def seed_everything(seed):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)

seed_everything(SEED)
assert CROP % 32 == 0

In [ ]:
# ---- Đọc dữ liệu Dublin ----
def find_file(name, *bases):
    for base in bases:
        hits = glob.glob(os.path.join(base, '**', name), recursive=True)
        if hits:
            return hits[0]
    raise FileNotFoundError(f'Không thấy {name}: kiểm tra Add Input.')


ANN_PATH = find_file('annotations.json', INPUT_DIR)
DUB_ROOT = os.path.dirname(ANN_PATH)
with open(ANN_PATH, encoding='utf-8') as fh:
    ann = json.load(fh)
DW, DH = ann['width'], ann['height']


def ignore_mask(polys, h, w):
    img = Image.new('L', (w, h), 0)
    for poly in polys:
        ImageDraw.Draw(img).polygon([tuple(p) for p in poly], fill=1)
    return np.asarray(img, bool)


def to_small_valid(ign):
    # Ô 1/4 hợp lệ nếu dưới một nửa diện tích nằm trong vùng bỏ qua.
    h, w = ign.shape
    return ign.reshape(h // STRIDE, STRIDE, w // STRIDE, STRIDE).mean(axis=(1, 3)) < 0.5


dub = []
for r in ann['images']:
    pts = np.asarray(r['points'], np.float32).reshape(-1, 2)
    ign = ignore_mask(r['ignore'], DH, DW)
    if len(pts):
        xi, yi = np.clip(pts[:, 0].astype(int), 0, DW - 1), np.clip(pts[:, 1].astype(int), 0, DH - 1)
        pts = pts[~ign[yi, xi]]            # điểm trong vùng bỏ qua không tính
    day, hour = r['utc'][:10], int(r['utc'][11:13])
    dub.append({'name': r['image'], 'path': os.path.join(DUB_ROOT, 'images', r['image']), 'points': pts,
                'valid': to_small_valid(ign), 'day': day, 'night': not (NIGHT_HOURS[0] <= hour < NIGHT_HOURS[1]),
                'w': DW, 'h': DH})
split_of = lambda d: 'test' if d in TEST_DAYS else 'val' if d in VAL_DAYS else 'train'
dub_split = {s: [r for r in dub if split_of(r['day']) == s] for s in ('train', 'val', 'test')}
if QUICK:
    dub_split = {s: v[:8] for s, v in dub_split.items()}

rows = {s: {'số ảnh': len(v), 'số người': int(sum(len(r['points']) for r in v)),
            'trung bình/ảnh': round(np.mean([len(r['points']) for r in v]), 1),
            'ảnh ban đêm': int(sum(r['night'] for r in v)), 'ngày': ', '.join(sorted({r['day'][5:] for r in v}))}
        for s, v in dub_split.items()}
display(pd.DataFrame(rows).T)


# ---- Đọc ShanghaiTech B (để trộn vào batch và kiểm tra quên) ----
def find_shb_root():
    for dirpath, dirnames, _ in os.walk(INPUT_DIR):
        if 'part_B' in dirnames:
            return dirpath
    raise FileNotFoundError('Không thấy ShanghaiTech: kiểm tra Add Input.')


SHB_ROOT = find_shb_root()
natural_key = lambda p: [int(t) if t.isdigit() else t for t in re.split(r'(\d+)', os.path.basename(p))]


def load_shb(split):
    recs = []
    for path in sorted(glob.glob(os.path.join(SHB_ROOT, 'part_B', split, 'images', '*.jpg')), key=natural_key):
        base = os.path.dirname(os.path.dirname(path))
        gt_dir = next(d for d in (os.path.join(base, n) for n in ('ground-truth', 'ground_truth', 'gt'))
                      if os.path.isdir(d))
        mat = os.path.join(gt_dir, 'GT_' + os.path.splitext(os.path.basename(path))[0] + '.mat')
        pts = np.asarray(sio.loadmat(mat)['image_info'][0, 0][0, 0][0], np.float32).reshape(-1, 2)
        w, h = Image.open(path).size
        recs.append({'name': os.path.basename(path), 'path': path, 'points': pts, 'w': w, 'h': h})
    return recs


shb_train, shb_test = load_shb('train_data'), load_shb('test_data')
if QUICK:
    shb_train, shb_test = shb_train[:16], shb_test[:8]
print(f'ShanghaiTech B: {len(shb_train)} ảnh train, {len(shb_test)} ảnh test | Dublin: {len(dub)} ảnh')

In [ ]:
# ---- Nhãn density: Gaussian tổng bằng 1 quanh mỗi điểm, gộp 1/4 ----
def gaussian_kernel(sigma):
    r = int(math.ceil(3 * sigma))
    ax = np.arange(-r, r + 1, dtype=np.float32)
    g = np.exp(-ax ** 2 / (2 * sigma ** 2))
    k = np.outer(g, g)
    return (k / k.sum()).astype(np.float32)


def make_density(points, h, w, kernel):
    den = np.zeros((h, w), np.float32)
    r = kernel.shape[0] // 2
    for x, y in points:
        x, y = min(max(int(x), 0), w - 1), min(max(int(y), 0), h - 1)
        y0, y1, x0, x1 = max(y - r, 0), min(y + r + 1, h), max(x - r, 0), min(x + r + 1, w)
        patch = kernel[y0 - y + r:y1 - y + r, x0 - x + r:x1 - x + r]
        den[y0:y1, x0:x1] += patch / patch.sum()
    return den


def sum_pool(den, s=STRIDE):
    h, w = den.shape
    H, W = math.ceil(h / s) * s, math.ceil(w / s) * s
    den = np.pad(den, ((0, H - h), (0, W - w)))
    return den.reshape(H // s, s, W // s, s).sum(axis=(1, 3))


def small_density(recs, sigma):
    k = gaussian_kernel(sigma)
    return {r['name']: sum_pool(make_density(r['points'], r['h'], r['w'], k)) for r in recs}


t0 = time.time()
SHB_DEN = small_density(shb_train, SHB_SIGMA)
print(f'Density ShanghaiTech B: {time.time() - t0:.0f}s')

MEAN = np.array([0.485, 0.456, 0.406], np.float32)
STD = np.array([0.229, 0.224, 0.225], np.float32)


class Patches(Dataset):
    # (ảnh, density 1/4 × DENSITY_SCALE, mask "có người" 1/4, vùng hợp lệ 1/4). Cắt theo ô 1/4 để khớp nhãn.
    def __init__(self, recs, dens, repeat=1):
        self.recs, self.dens, self.repeat = recs, dens, repeat

    def __len__(self):
        return len(self.recs) * self.repeat

    def __getitem__(self, i):
        r = self.recs[i % len(self.recs)]
        den = self.dens[r['name']]
        valid = r.get('valid')
        valid = np.ones_like(den, bool) if valid is None else valid
        cs = CROP // STRIDE
        ys, xs = random.randint(0, max(den.shape[0] - cs, 0)), random.randint(0, max(den.shape[1] - cs, 0))
        img = np.asarray(Image.open(r['path']).convert('RGB'), np.float32) / 255.0
        img = (img[ys * STRIDE:ys * STRIDE + CROP, xs * STRIDE:xs * STRIDE + CROP] - MEAN) / STD
        d, v = den[ys:ys + cs, xs:xs + cs], valid[ys:ys + cs, xs:xs + cs]
        if random.random() < 0.5:
            img, d, v = img[:, ::-1], d[:, ::-1], v[:, ::-1]
        t = lambda a: torch.from_numpy(np.ascontiguousarray(a, dtype=np.float32))
        return (t(img.transpose(2, 0, 1)), t(d[None] * DENSITY_SCALE), t((d > MASK_THRESH)[None]), t(v[None]))


def make_loader(ds, bs):
    return DataLoader(ds, batch_size=bs, shuffle=True, drop_last=True, num_workers=NUM_WORKERS,
                      pin_memory=device.type == 'cuda', persistent_workers=NUM_WORKERS > 0)

In [ ]:
class ChainedResidualPool(nn.Module):
    def __init__(self, channels, n_stages=4):
        super().__init__()
        self.pool = nn.MaxPool2d(5, stride=1, padding=2)
        self.convs = nn.ModuleList([nn.Conv2d(channels, channels, 1, bias=False) for _ in range(n_stages)])

    def forward(self, x):
        top = x
        for conv in self.convs:
            top = conv(self.pool(top))
            x = x + top
        return x


class MobileCount(nn.Module):
    # Giống hệt notebook train gốc (final_mobilecount_cv.ipynb); mask=True là bản có nhánh mask.
    def __init__(self, pretrained=True, mask=False, channels=64):
        super().__init__()
        weights = torchvision.models.MobileNet_V2_Weights.IMAGENET1K_V1 if pretrained else None
        f = torchvision.models.mobilenet_v2(weights=weights).features
        self.encoder = nn.ModuleList([f[:4], f[4:7], f[7:14], f[14:18]])
        enc_ch = (24, 32, 96, 320)
        self.project = nn.ModuleList([nn.Conv2d(c, channels, 1, bias=False) for c in enc_ch[:3]])
        self.deep = nn.Sequential(nn.Conv2d(enc_ch[3], channels, 1, bias=False), nn.ReLU(inplace=True))
        self.crp = nn.ModuleList([ChainedResidualPool(channels) for _ in range(4)])
        self.fuse = nn.ModuleList([nn.Conv2d(channels, channels, 1, bias=False) for _ in range(3)])
        self.head = nn.Sequential(nn.Conv2d(channels, channels, 3, padding=1), nn.ReLU(inplace=True),
                                  nn.Conv2d(channels, 1, 1))
        self.mask_head = None
        if mask:
            self.mask_head = nn.Sequential(nn.Conv2d(channels, 32, 3, padding=1), nn.ReLU(inplace=True),
                                           nn.Conv2d(32, 1, 1))

    def forward(self, x, with_mask=False):
        feats = []
        for stage in self.encoder:
            x = stage(x)
            feats.append(x)
        y = self.crp[3](self.deep(feats[3]))
        for i in (2, 1, 0):
            y = F.interpolate(self.fuse[i](y), size=feats[i].shape[-2:], mode='bilinear', align_corners=False)
            y = self.crp[i](F.relu(self.project[i](feats[i]) + y))
        den = self.head(y)
        if self.mask_head is None:
            return (den, None) if with_mask else den
        logit = self.mask_head(y)
        den = den * torch.sigmoid(logit)
        return (den, logit) if with_mask else den


def unwrap(state):
    return state.get('model', state) if isinstance(state, dict) and 'model' in state else state


def find_checkpoint():
    cands = glob.glob(os.path.join(INPUT_DIR, '**', '*.pth'), recursive=True)
    for p in cands:
        if CKPT_NAME_HINT in os.path.basename(p):
            return p
    for p in cands:
        if any(k.startswith('mask_head') for k in unwrap(torch.load(p, map_location='cpu'))):
            return p
    raise FileNotFoundError('Không thấy checkpoint mask: Add Input model chứa file .pth.')


CKPT = find_checkpoint()
print('Checkpoint gốc:', CKPT)


def load_base():
    model = MobileCount(pretrained=False, mask=True).to(device)
    model.load_state_dict(unwrap(torch.load(CKPT, map_location=device)))
    return model

In [ ]:
@torch.no_grad()
def predict(model, path, valid=None):
    # Density 1/4 (đã chia DENSITY_SCALE) và số người dự đoán (chỉ cộng ô hợp lệ nếu có vùng bỏ qua).
    img = (np.asarray(Image.open(path).convert('RGB'), np.float32) / 255.0 - MEAN) / STD
    h, w, _ = img.shape
    H, W = math.ceil(h / 32) * 32, math.ceil(w / 32) * 32
    img = np.pad(img, ((0, H - h), (0, W - w), (0, 0)))
    x = torch.from_numpy(np.ascontiguousarray(img.transpose(2, 0, 1)))[None].to(device)
    den = model(x)[0, 0].float().cpu().numpy()[:math.ceil(h / STRIDE), :math.ceil(w / STRIDE)] / DENSITY_SCALE
    return den, (float(den[valid].sum()) if valid is not None else float(den.sum()))


def evaluate(model, recs):
    model.eval()
    rows = []
    for r in recs:
        _, pred = predict(model, r['path'], r.get('valid'))
        rows.append({'name': r['name'], 'gt': len(r['points']), 'pred': pred,
                     'night': r.get('night', False), 'day': r.get('day', '')})
    model.train()
    df = pd.DataFrame(rows)
    df['error'] = df['pred'] - df['gt']
    return df


def mae_mse(df):
    return float(df['error'].abs().mean()), float(np.sqrt((df['error'] ** 2).mean()))


def report(model, label):
    out = {'label': label}
    for split in ('val', 'test'):
        df = evaluate(model, dub_split[split])
        out[f'Dublin {split} MAE'], out[f'Dublin {split} MSE'] = mae_mse(df)
        if split == 'test':
            out['_df'] = df
            for tag, sel in (('ngày', ~df['night']), ('đêm', df['night'])):
                out[f'Dublin test MAE {tag}'] = float(df[sel]['error'].abs().mean()) if sel.any() else np.nan
    out['SHB test MAE'], out['SHB test MSE'] = mae_mse(evaluate(model, shb_test))
    return out


def show(*reports):
    display(pd.DataFrame([{k: v for k, v in r.items() if k != '_df'} for r in reports]).set_index('label').T.round(2))


base = load_base()
BEFORE = report(base, 'Trước fine-tune')
del base
show(BEFORE)

In [ ]:
def masked_mean(x, w):
    return (x * w).sum() / w.sum().clamp(min=1.0)


def finetune(sigma, tag):
    print(f'--- Fine-tune với sigma = {sigma} ---')
    seed_everything(SEED)
    dub_dens = small_density(dub_split['train'], sigma)
    dub_loader = make_loader(Patches(dub_split['train'], dub_dens, REPEAT), DUB_BS)
    shb_loader = make_loader(Patches(shb_train, SHB_DEN), SHB_BS) if SHB_BS else None
    shb_iter = iter(shb_loader) if shb_loader else None

    model = load_base()
    optimizer = torch.optim.Adam(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS, eta_min=LR_MIN)
    use_amp = device.type == 'cuda'
    scaler = torch.amp.GradScaler('cuda', enabled=use_amp)
    best_mae, best_epoch, best_state, history, start = float('inf'), 0, None, [], time.time()

    for epoch in range(1, EPOCHS + 1):
        for p in model.encoder.parameters():
            p.requires_grad = epoch > FREEZE_EPOCHS
        model.train()
        losses = []
        for img, den, mask_gt, valid in dub_loader:
            if shb_iter is not None:
                try:
                    extra = next(shb_iter)
                except StopIteration:
                    shb_iter = iter(shb_loader)
                    extra = next(shb_iter)
                img, den, mask_gt, valid = (torch.cat([a, b]) for a, b in zip((img, den, mask_gt, valid), extra))
            img, den, mask_gt, valid = (t.to(device, non_blocking=True) for t in (img, den, mask_gt, valid))
            with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=use_amp):
                out, logit = model(img, with_mask=True)
            loss = masked_mean((out.float() - den) ** 2, valid)
            loss = loss + MASK_LAMBDA * masked_mean(
                F.binary_cross_entropy_with_logits(logit.float(), mask_gt, reduction='none'), valid)
            optimizer.zero_grad(set_to_none=True)
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
            losses.append(loss.item())
        scheduler.step()
        row = {'epoch': epoch, 'lr': optimizer.param_groups[0]['lr'], 'loss': float(np.mean(losses))}
        if epoch % VAL_EVERY == 0 or epoch == EPOCHS:
            row['val_MAE'], row['val_MSE'] = mae_mse(evaluate(model, dub_split['val']))
            if row['val_MAE'] < best_mae:
                best_mae, best_epoch = row['val_MAE'], epoch
                best_state = copy.deepcopy(model.state_dict())
            print(f"  epoch {epoch:3d} | loss {row['loss']:.5f} | val MAE {row['val_MAE']:6.2f} "
                  f"(tốt nhất {best_mae:.2f} ở epoch {best_epoch}) | {(time.time() - start) / 60:5.1f} phút")
        history.append(row)
    del dub_loader, shb_loader, optimizer
    if device.type == 'cuda':
        torch.cuda.empty_cache()
    torch.save(best_state, os.path.join(WORK_DIR, f'mask_dublin_sigma{sigma}.pth'))
    pd.DataFrame(history).to_csv(os.path.join(WORK_DIR, f'history_sigma{sigma}.csv'), index=False)
    return {'sigma': sigma, 'val_MAE': best_mae, 'best_epoch': best_epoch, 'state': best_state,
            'history': pd.DataFrame(history), 'minutes': (time.time() - start) / 60}


RUNS = [finetune(s, f'sigma{s}') for s in SIGMAS]
display(pd.DataFrame([{k: v for k, v in r.items() if k not in ('state', 'history')} for r in RUNS]).set_index('sigma').round(2))

In [ ]:
# Chọn sigma theo val MAE, rồi đo test một lần duy nhất.
best = min(RUNS, key=lambda r: r['val_MAE'])
model = MobileCount(pretrained=False, mask=True).to(device)
model.load_state_dict(best['state'])
AFTER = report(model, f"Sau fine-tune (sigma {best['sigma']}, epoch {best['best_epoch']})")
show(BEFORE, AFTER)

dfb, dfa = BEFORE['_df'], AFTER['_df']
pd.DataFrame({'name': dfb['name'], 'gt': dfb['gt'], 'night': dfb['night'],
              'pred_before': dfb['pred'], 'pred_after': dfa['pred']}).to_csv(
    os.path.join(WORK_DIR, 'test_predictions.csv'), index=False)

fig, axes = plt.subplots(1, 3, figsize=(18, 4.8))
for ax, df, title in ((axes[0], dfb, 'Trước'), (axes[1], dfa, 'Sau')):
    ax.scatter(df['gt'], df['pred'], s=18, c=np.where(df['night'], 'tab:purple', 'tab:orange'))
    lim = max(df['gt'].max(), df['pred'].max()) * 1.05
    ax.plot([0, lim], [0, lim], 'k--', lw=1)
    ax.set(title=f"{title}: MAE {df['error'].abs().mean():.2f}", xlabel='số người thật', ylabel='dự đoán')
axes[0].scatter([], [], c='tab:orange', label='ban ngày'); axes[0].scatter([], [], c='tab:purple', label='ban đêm')
axes[0].legend()
for r in RUNS:
    h = r['history'].dropna(subset=['val_MAE'])
    axes[2].plot(h['epoch'], h['val_MAE'], marker='o', label=f"sigma {r['sigma']}")
axes[2].axhline(BEFORE['Dublin val MAE'], color='gray', ls='--', label='trước fine-tune')
axes[2].set(title='Val MAE theo epoch', xlabel='epoch', ylabel='MAE (người)'); axes[2].legend()
plt.tight_layout(); plt.savefig(os.path.join(WORK_DIR, 'finetune_result.png'), bbox_inches='tight'); plt.show()

# Ảnh mẫu: density dự đoán sau fine-tune trên 4 ảnh test (vùng bỏ qua tô xám)
base = load_base()
picks = [dub_split['test'][i] for i in np.linspace(0, len(dub_split['test']) - 1, 4).astype(int)]
fig, axes = plt.subplots(2, 4, figsize=(20, 6.5))
for j, r in enumerate(picks):
    for i, (m, tag) in enumerate(((base, 'trước'), (model, 'sau'))):
        den, pred = predict(m, r['path'], r['valid'])
        axes[i, j].imshow(Image.open(r['path'])); axes[i, j].imshow(np.where(r['valid'], den, np.nan), cmap='jet',
            alpha=0.5, extent=(0, DW, DH, 0))
        axes[i, j].set_title(f"{tag}: dự đoán {pred:.0f} / thật {len(r['points'])}"); axes[i, j].axis('off')
plt.tight_layout(); plt.savefig(os.path.join(WORK_DIR, 'finetune_samples.png'), bbox_inches='tight'); plt.show()
del base

In [ ]:
# Lưu kết quả cuối: trọng số để đưa vào demo + bảng số liệu
torch.save(model.state_dict(), os.path.join(WORK_DIR, 'mask_dublin.pth'))
summary = {'sigma': best['sigma'], 'best_epoch': best['best_epoch'], 'checkpoint_goc': os.path.basename(CKPT),
           'before': {k: v for k, v in BEFORE.items() if k != '_df'},
           'after': {k: v for k, v in AFTER.items() if k != '_df'},
           'sweep': [{'sigma': r['sigma'], 'val_MAE': r['val_MAE'], 'best_epoch': r['best_epoch']} for r in RUNS],
           'config': {'epochs': EPOCHS, 'repeat': REPEAT, 'dub_bs': DUB_BS, 'shb_bs': SHB_BS, 'lr': LR, 'lr_min': LR_MIN,
                      'freeze_epochs': FREEZE_EPOCHS, 'crop': CROP, 'test_days': TEST_DAYS, 'val_days': VAL_DAYS,
                      'n_train': len(dub_split['train']), 'n_val': len(dub_split['val']),
                      'n_test': len(dub_split['test'])}, 'device': GPU_NAME, 'quick': QUICK}
with open(os.path.join(WORK_DIR, 'finetune_results.json'), 'w', encoding='utf-8') as fh:
    json.dump(summary, fh, ensure_ascii=False, indent=2)
for f in glob.glob(os.path.join(WORK_DIR, 'mask_dublin_sigma*.pth')):
    os.remove(f)     # chỉ giữ mask_dublin.pth để thư mục output gọn
print('Đã lưu mask_dublin.pth, finetune_results.json, test_predictions.csv, ảnh kết quả.')
print(f"Dublin test MAE: {BEFORE['Dublin test MAE']:.2f} → {AFTER['Dublin test MAE']:.2f} | "
      f"ShanghaiTech B test MAE: {BEFORE['SHB test MAE']:.2f} → {AFTER['SHB test MAE']:.2f}")